# USDA Nutrition Exploratory Data Analysis

This notebook explores the USDA FoodData Central database. We will load the nutritional facts, review the calorie and macronutrient distributions, check for missing data, and handle any gaps to ensure a clean dataset moving forward.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_theme(style="whitegrid")

## 1. Load USDA CSVs
Assuming the FoodData Central database is unzipped into `../data/usda/` containing `food.csv` and `food_nutrient.csv`.

In [ ]:
DATA_DIR = "../data/usda"
FOOD_CSV = os.path.join(DATA_DIR, "food.csv")
NUTRIENT_CSV = os.path.join(DATA_DIR, "food_nutrient.csv")

if os.path.exists(FOOD_CSV) and os.path.exists(NUTRIENT_CSV):
    food_df = pd.read_csv(FOOD_CSV)
    nutrient_df = pd.read_csv(NUTRIENT_CSV, low_memory=False)
    print(f"Loaded {len(food_df)} foods and {len(nutrient_df)} nutrient records.")
else:
    print("USDA CSVs not found. Please download from FoodData Central, unzip into `data/usda/`.")
    # For structural progress, let's create a simulated dummy dataframe.
    print("Using simulated data for notebook execution.")
    food_df = pd.DataFrame({'fdc_id': [1, 2, 3], 'description': ['Apple', 'Chicken Breast', 'Rice']})
    nutrient_df = pd.DataFrame({
        'fdc_id': [1, 1, 1, 2, 2, 2, 3, 3, 3],
        'nutrient_id': [1008, 1003, 1004, 1008, 1003, 1004, 1008, 1003, 1004],
        'amount': [52, 0.3, 0.2, 165, 31, 3.6, 130, 2.7, 0.3]
    })

## 2. Combine and Filter Nutrients
Key Nutrient IDs:
- 1008: Calories (kcal)
- 1003: Protein (g)
- 1004: Total Fat (g)
- 1005: Carbohydrates (g)
- 1079: Fiber (g)
- 2000: Sugars (g)

In [ ]:
TARGET_NUTRIENTS = {
    1008: 'Calories_kcal',
    1003: 'Protein_g',
    1004: 'Fat_g',
    1005: 'Carbs_g',
    1079: 'Fiber_g',
    2000: 'Sugar_g'
}

# Filter nutrient dataframe to only our targets
filtered_nutrients = nutrient_df[nutrient_df['nutrient_id'].isin(TARGET_NUTRIENTS.keys())].copy()
filtered_nutrients['nutrient_name'] = filtered_nutrients['nutrient_id'].map(TARGET_NUTRIENTS)

# Pivot table to have one row per food item
pivoted_nutrients = filtered_nutrients.pivot_table(
    index='fdc_id', 
    columns='nutrient_name', 
    values='amount', 
    aggfunc='first'
).reset_index()

# Merge with food names
food_macros = pd.merge(food_df[['fdc_id', 'description']], pivoted_nutrients, on='fdc_id', how='inner')
food_macros.head()

## 3. Check Nutrient Completeness & Handle Missing Data

In [ ]:
print("Missing values per column:")
print(food_macros.isna().sum())

# Handle missing values by filling with 0 (since missing macro/calorie often implies 0 in basic DBs)
# Or drop completely empty entries
food_macros.fillna(0, inplace=True)

print("\nAfter imputation:")
print(food_macros.isna().sum())

## 4. Visualizing Distributions

In [ ]:
# Plotting histograms for Calories, Protein, Fat, Carbs
macros_to_plot = ['Calories_kcal', 'Protein_g', 'Fat_g', 'Carbs_g']
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for i, macro in enumerate(macros_to_plot):
    if macro in food_macros.columns:
        ax = axes[i//2, i%2]
        # filtering outliers for visualization purposes
        q99 = food_macros[macro].quantile(0.99) if not food_macros.empty else 100
        sns.histplot(food_macros[food_macros[macro] < q99][macro], bins=30, ax=ax)
        ax.set_title(f'Distribution of {macro}')

plt.tight_layout()
plt.show()